The preprocessing piece of the ml will take the IMU data and make sure it is clean enough to segment into strokes and extract features from. Functions live in `imu_preprocessing.py` so later notebooks can import them.

**What "clean" means here**
1. **Timing**: monotonic timestamps on a uniform 100 Hz grid. Short gaps (≤ 50 ms) are interpolated; longer gaps become NaN and split the recording into `segment`s, so filters never run across a dropout.
2. **Integrity**: no NaNs, no stuck/flatlined channels, no saturation at the sensor's full-scale range.
3. **Spikes**: isolated single-sample glitches replaced (Hampel filter, conservative, so real catch impacts survive).
4. **Bias**: gyro bias removed if the recording contains ≥ 1 s of stillness, or if supplied from a separate static recording.
5. **Noise**: zero-phase 4th-order Butterworth low-pass. Accel/gyro at 12 Hz (rowing signal: stroke ≈ 0.6 Hz, catch impact has content up to ~12 Hz); mag at 5 Hz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal

import imu_preprocessing as ip

FS = 100.0
DATA_DIR = Path("../test data/imu")
OUT_DIR = Path("../test data/imu_clean")
files = sorted(DATA_DIR.glob("*.csv"))
files

## 1. Quality audit of the raw file
`acc_range_g` / `gyr_range_rads`: set these to the sensor's configured full-scale range to enable the saturation check.

In [ ]:
path = files[0]
# load_imu auto-detects column names / units and keeps only the 6 DoF
# (accel + gyro). Override with columns={...}, acc_unit="g", gyr_unit="degs", fs=...
raw = ip.load_imu(path, verbose=True)
report = ip.quality_report(raw, fs_nominal=FS, acc_range_g=None, gyr_range_rads=None)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, cols, unit in zip(axes, [ip.ACC, ip.GYR], ["m/s²", "rad/s"]):
    for c in cols:
        ax.plot(raw.t_s, raw[c], lw=0.8, label=c)
    ax.set_ylabel(unit); ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Time (s)")
axes[0].set_title(f"Raw: {path.name}")
plt.tight_layout()

## 2. Choosing the low-pass cutoff
Cumulative power shows how much of each signal sits below a given frequency. Keep the cutoff above where real motion ends (~99% of power) and below the noise floor.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for c in ip.ACC + ip.GYR:
    x = raw[c].to_numpy()
    f, P = signal.welch(x - x.mean(), fs=FS, nperseg=1024)
    ax.plot(f, np.cumsum(P) / P.sum(), label=c)
ax.axvline(12, color="k", ls="--", lw=1)
ax.set_xlim(0, 30); ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Cumulative power")
ax.legend(fontsize=8, ncol=2); plt.tight_layout()

## 3. Clean

In [ ]:
clean, info = ip.clean_imu(raw, fs=FS, acc_cutoff=12, gyr_cutoff=12)
info

In [ ]:
t0, t1 = 5, 10   # zoom window
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, cols in zip(axes, [ip.ACC, ip.GYR]):
    for c in cols:
        r = raw[(raw.t_s >= t0) & (raw.t_s <= t1)]
        k = clean[(clean.t_s >= t0) & (clean.t_s <= t1)]
        line, = ax.plot(r.t_s, r[c], lw=0.6, alpha=0.4)
        ax.plot(k.t_s, k[c], lw=1.4, color=line.get_color(), label=c)
    ax.legend(loc="upper right", fontsize=8)
axes[0].set_ylabel("m/s²"); axes[1].set_ylabel("rad/s"); axes[1].set_xlabel("Time (s)")
axes[0].set_title("Raw (faint) vs clean")
plt.tight_layout()

## 4. Sanity check: does the pipeline catch real faults?
Inject a dropout, a spike, a duplicate row and a stuck channel into a copy of the file and confirm the audit flags them and cleaning handles them.

In [ ]:
bad = raw.drop(index=range(500, 530)).copy()          # 300 ms dropout
bad.loc[100, "ax_mps2"] = 150                          # spike
bad = pd.concat([bad, bad.iloc[[10]]])                 # duplicate row
bad.loc[1200:1210, "gy_rads"] = bad.loc[1200, "gy_rads"]  # stuck channel
_ = ip.quality_report(bad, fs_nominal=FS)
bad_clean, bad_info = ip.clean_imu(bad, fs=FS)
print(bad_info)
print("max ax after cleaning:", round(bad_clean.ax_mps2.max(), 2))

## 5. Batch-clean every file and save

In [ ]:
# Batch: works on any rower's IMU CSVs (column names / units auto-detected)
summary = ip.preprocess_folder(DATA_DIR, OUT_DIR, fs_out=FS)
summary

## Open issues
- **Magnetometer**: magnitude varies ~43% (Earth's field should be constant), likely interference from the erg's flywheel/frame or no hard/soft-iron calibration. It also updates slower than 100 Hz (sample-and-hold). Don't use it for heading unless it's calibrated away from the erg.
- **Gyro bias**: this file has no still period, so bias isn't removed. Record ~5 s with the sensor sitting still at the start of each session (or a separate static file and pass `gyro_bias=`).
- **Saturation**: set `acc_range_g` / `gyr_range_rads` once the sensor's configured ranges are known. Peaks here are ~2 g and ~8.4 rad/s.
- **Multiple IMUs**: for full-body work, each sensor's file needs time-syncing to the others (and to the erg CSV) before cleaning results can be combined.